# Exercițiul 1 — Două zaruri

Spațiul de probabilitate: $\Omega=\{(x,y): x,y\in\{1,\dots,6\}\}$, $|\Omega|=36$, toate rezultatele echiprobabile.

**1.** $A=\{X+Y\ge 10\}$, $B=\{X \text{ par}\}$.

$|B|=3\cdot 6=18$. Cazuri favorabile pentru $A\cap B$: $(4,6),(6,4),(6,5),(6,6)$, deci $|A\cap B|=4$.

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}=\frac{4/36}{18/36}=\frac{4}{18}=\frac{2}{9}\approx 0{,}2222$$

**2.** $C=\{\text{cel puțin un zar par}\}$. Complementara: ambele zaruri impare, $3\cdot 3=9$ cazuri, deci $|C|=36-9=27$.

$A=\{(4,6),(6,4),(5,5),(5,6),(6,5),(6,6)\}$, $|A|=6$. Dintre acestea, $(5,5)$ nu are niciun zar par, deci $|A\cap C|=5$.

$$P(A\mid C)=\frac{5/36}{27/36}=\frac{5}{27}\approx 0{,}1852$$

**3.** $D=\{X+Y=8\}=\{(2,6),(3,5),(4,4),(5,3),(6,2)\}$, $|D|=5$. $E=\{\text{ambele pare}\}$; $D\cap E=\{(2,6),(4,4),(6,2)\}$, $|D\cap E|=3$.

$$P(E\mid D)=\frac{3/36}{5/36}=\frac{3}{5}=0{,}6$$

**4.** $P(A)=\dfrac{6}{36}=\dfrac16$, $P(B)=\dfrac12$, $P(A\cap B)=\dfrac{4}{36}=\dfrac19$.

$$P(A)\,P(B)=\frac16\cdot\frac12=\frac1{12}\neq\frac19=P(A\cap B)$$

Deci evenimentele $A$ și $B$ **nu sunt independente**.

In [1]:
from fractions import Fraction
from itertools import product

omega = list(product(range(1, 7), repeat=2))

def P(event, cond=lambda w: True):
    space = [w for w in omega if cond(w)]
    fav = [w for w in space if event(w)]
    return Fraction(len(fav), len(space))

A = lambda w: w[0] + w[1] >= 10
B = lambda w: w[0] % 2 == 0
C = lambda w: w[0] % 2 == 0 or w[1] % 2 == 0
D = lambda w: w[0] + w[1] == 8
E = lambda w: w[0] % 2 == 0 and w[1] % 2 == 0

exact = {
    "P(A | X par)": P(A, B),
    "P(A | cel puțin un zar par)": P(A, C),
    "P(ambele pare | suma = 8)": P(E, D),
}
for k, v in exact.items():
    print(f"{k} = {v} ≈ {float(v):.4f}")

pA, pB = P(A), P(B)
pAB = P(lambda w: A(w) and B(w))
print(f"P(A)P(B) = {pA * pB}, P(A ∩ B) = {pAB}, independente: {pA * pB == pAB}")

P(A | X par) = 2/9 ≈ 0.2222
P(A | cel puțin un zar par) = 5/27 ≈ 0.1852
P(ambele pare | suma = 8) = 3/5 ≈ 0.6000
P(A)P(B) = 1/12, P(A ∩ B) = 1/9, independente: False


## Extensie Python — simulare cu 100.000 de aruncări

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N = 100_000
X = rng.integers(1, 7, size=N)
Y = rng.integers(1, 7, size=N)

a = (X + Y) >= 10
b = (X % 2 == 0)
c = (X % 2 == 0) | (Y % 2 == 0)
d = (X + Y) == 8
e = (X % 2 == 0) & (Y % 2 == 0)

estim = {
    "P(A | X par)": a[b].mean(),
    "P(A | cel puțin un zar par)": a[c].mean(),
    "P(ambele pare | suma = 8)": e[d].mean(),
}

tabel = pd.DataFrame({
    "Valoare exactă": [float(v) for v in exact.values()],
    "Estimare (simulare)": list(estim.values()),
}, index=list(exact.keys()))
tabel["Eroare absolută"] = (tabel["Valoare exactă"] - tabel["Estimare (simulare)"]).abs()
tabel.round(4)

,Valoare exactă,Estimare (simulare),Eroare absolută
P(A | X par),0.2222,0.2241,0.0018
P(A | cel puțin un zar par),0.1852,0.1854,0.0002
P(ambele pare | suma = 8),0.6000,0.5965,0.0035


In [3]:
print(f"P(A) estimat = {a.mean():.4f}, P(B) estimat = {b.mean():.4f}")
print(f"P(A)P(B) estimat = {a.mean() * b.mean():.4f}, P(A ∩ B) estimat = {(a & b).mean():.4f}")

P(A) estimat = 0.1665, P(B) estimat = 0.4996
P(A)P(B) estimat = 0.0832, P(A ∩ B) estimat = 0.1119


# Exercițiul 2 — Filtrarea mesajelor spam

Notăm $S$ = „mesaj spam”, $L$ = „mesaj legitim”, $O$ = „conține cuvântul ofertă”, $G$ = „conține cuvântul gratuit”.

Date: $P(S)=0{,}3$, $P(L)=0{,}7$, $P(O\mid S)=0{,}8$, $P(O\mid L)=0{,}15$, $P(G\mid S)=0{,}6$, $P(G\mid L)=0{,}1$.

**1.** Din formula probabilității totale și formula lui Bayes:

$$P(O)=P(O\mid S)P(S)+P(O\mid L)P(L)=0{,}8\cdot 0{,}3+0{,}15\cdot 0{,}7=0{,}24+0{,}105=0{,}345$$

$$P(S\mid O)=\frac{P(O\mid S)P(S)}{P(O)}=\frac{0{,}24}{0{,}345}=\frac{16}{23}\approx 0{,}6957$$

**2.** Cu independență condiționată (față de clasă): $P(O\cap G\mid S)=P(O\mid S)P(G\mid S)$ și analog pentru $L$.

$$P(O\cap G\mid S)=0{,}8\cdot 0{,}6=0{,}48,\qquad P(O\cap G\mid L)=0{,}15\cdot 0{,}1=0{,}015$$

$$P(O\cap G)=0{,}48\cdot 0{,}3+0{,}015\cdot 0{,}7=0{,}144+0{,}0105=0{,}1545$$

$$P(S\mid O\cap G)=\frac{0{,}144}{0{,}1545}=\frac{96}{103}\approx 0{,}9320$$

**3.** Ipoteza $P(O\cap G\mid S)=P(O\mid S)P(G\mid S)$ presupune că, odată cunoscută clasa mesajului, apariția unui cuvânt nu aduce nicio informație despre apariția celuilalt. În realitate cuvintele sunt corelate chiar și în interiorul aceleiași clase: mesajele spam sunt construite după șabloane (de exemplu „ofertă” și „gratuit” apar frecvent împreună în aceeași formulare promoțională), iar cuvintele depind de subiect, de limbă și de stilul autorului. Dacă $P(G\mid S,O)>P(G\mid S)$, produsul $P(O\mid S)P(G\mid S)$ **subestimează** probabilitatea reală a apariției comune în spam, iar dovezile se numără de două ori; probabilitatea posterioară calculată este atunci exagerat de apropiată de 1 (clasificatorul devine suprasigur).

**4.** Regulă de clasificare (decizie bayesiană cu prag): pentru un mesaj cu setul de cuvinte $W$, se calculează $P(S\mid W)$ și

$$\text{clasificare}(W)=\begin{cases}\text{spam}, & P(S\mid W)\ge \tau\\ \text{legitim}, & P(S\mid W)<\tau\end{cases}$$

Cu costuri egale ale erorilor, $\tau=0{,}5$. Dacă marcarea greșită a unui mesaj legitim ca spam costă $c_{L}$, iar trecerea unui spam drept legitim costă $c_{S}$, pragul optim este $\tau=\dfrac{c_{L}}{c_{L}+c_{S}}$ (de exemplu $\tau=0{,}9$ dacă pierderea unui mesaj legitim este de 9 ori mai gravă). Echivalent, se folosește raportul de verosimilitate $\dfrac{P(S\mid W)}{P(L\mid W)}\ge \dfrac{\tau}{1-\tau}$.

Pentru datele de mai sus, cu $\tau=0{,}5$: un mesaj cu „ofertă” ($0{,}6957$) este spam, iar unul cu „ofertă” și „gratuit” ($0{,}9320$) este spam.

In [4]:
pS, pL = 0.3, 0.7
pO_S, pO_L = 0.8, 0.15
pG_S, pG_L = 0.6, 0.1

p_S_O = pO_S * pS / (pO_S * pS + pO_L * pL)
p_S_OG = pO_S * pG_S * pS / (pO_S * pG_S * pS + pO_L * pG_L * pL)
print(f"P(S | O) = {p_S_O:.4f}")
print(f"P(S | O ∩ G) = {p_S_OG:.4f}")

def clasifica(p_spam, prag=0.5):
    return "spam" if p_spam >= prag else "legitim"

print(clasifica(p_S_O), clasifica(p_S_OG))

P(S | O) = 0.6957
P(S | O ∩ G) = 0.9320
spam spam


# Exercițiul 3 — Paradoxul lui Simpson

Notăm $P$ = „student promovat”, $A$, $B$ = disciplina, $\Pi_1,\Pi_2$ = programul de studiu.

**1.** Ratele de promovare pe programe:

| Program | Disciplina A | Disciplina B |
|---|---|---|
| Programul 1 | $\dfrac{81}{90}=0{,}90$ | $\dfrac{19}{20}=0{,}95$ |
| Programul 2 | $\dfrac{1}{10}=0{,}10$ | $\dfrac{16}{80}=0{,}20$ |

**2.** În ambele programe, **disciplina B** are rata de promovare mai mare ($0{,}95>0{,}90$ în Programul 1 și $0{,}20>0{,}10$ în Programul 2).

**3.** Rata globală:

$$\text{A: }\frac{81+1}{90+10}=\frac{82}{100}=0{,}82,\qquad \text{B: }\frac{19+16}{20+80}=\frac{35}{100}=0{,}35$$

Global, **disciplina A** are rata mai mare ($0{,}82>0{,}35$), deci ordinea se inversează.

**4.** Rata globală este o medie ponderată a ratelor pe programe, iar ponderile sunt diferite pentru cele două discipline:

$$0{,}82=\frac{90}{100}\cdot 0{,}90+\frac{10}{100}\cdot 0{,}10,\qquad 0{,}35=\frac{20}{100}\cdot 0{,}95+\frac{80}{100}\cdot 0{,}20$$

Disciplina A este urmată în principal de studenții Programului 1 ($90\%$ dintre cei evaluați), program în care ratele sunt mari la ambele discipline. Disciplina B este urmată în principal de studenții Programului 2 ($80\%$), program cu rate mici la ambele discipline. Diferența de compoziție (programul este o variabilă de confuzie) pe care o are fiecare disciplină depășește avantajul lui B în interiorul fiecărui program, iar agregarea compară în fapt populații diferite.

**5.** În termeni de probabilități condiționate:

$$P(P\mid A,\Pi_1)<P(P\mid B,\Pi_1)\quad(0{,}90<0{,}95),\qquad P(P\mid A,\Pi_2)<P(P\mid B,\Pi_2)\quad(0{,}10<0{,}20)$$

dar

$$P(P\mid A)>P(P\mid B)\quad(0{,}82>0{,}35).$$

Din formula probabilității totale:

$$P(P\mid D)=P(P\mid D,\Pi_1)\,P(\Pi_1\mid D)+P(P\mid D,\Pi_2)\,P(\Pi_2\mid D),\qquad D\in\{A,B\}$$

Inversarea apare deoarece $P(\Pi_1\mid A)=0{,}9\neq P(\Pi_1\mid B)=0{,}2$: ponderile $P(\Pi_i\mid D)$ diferă între discipline, deci disciplina și programul nu sunt independente. Inegalitățile pe fiecare strat nu se păstrează la media ponderată cu ponderi diferite.

In [5]:
import pandas as pd

date = pd.DataFrame({
    "Program": ["Programul 1", "Programul 1", "Programul 2", "Programul 2"],
    "Disciplina": ["A", "B", "A", "B"],
    "Promovati": [81, 19, 1, 16],
    "Evaluati": [90, 20, 10, 80],
})
date["Rata"] = date["Promovati"] / date["Evaluati"]
print(date.pivot(index="Program", columns="Disciplina", values="Rata").round(4))

global_ = date.groupby("Disciplina")[["Promovati", "Evaluati"]].sum()
global_["Rata globala"] = global_["Promovati"] / global_["Evaluati"]
print()
print(global_.round(4))

pondere = date.pivot(index="Program", columns="Disciplina", values="Evaluati")
print()
print((pondere / pondere.sum()).round(2))

Disciplina     A     B
Program               
Programul 1  0.9  0.95
Programul 2  0.1  0.20

            Promovati  Evaluati  Rata globala
Disciplina                                   
A                  82       100          0.82
B                  35       100          0.35

Disciplina     A    B
Program              
Programul 1  0.9  0.2
Programul 2  0.1  0.8


## Extensie Python — exemplu propriu de paradox Simpson

Două medicamente, $X$ și $Y$, testate pe pacienți cu formă ușoară și formă gravă a bolii; „succes” = vindecare.

| Medicament | Formă ușoară | Formă gravă |
|---|---|---|
| $X$ | $19$ din $20$ | $40$ din $180$ |
| $Y$ | $180$ din $200$ | $3$ din $20$ |

$X$ are rata de succes mai mare în ambele forme ($0{,}95>0{,}90$ și $0{,}222>0{,}15$), dar global $Y$ are rata mai mare ($0{,}832>0{,}295$), deoarece $X$ a fost administrat în principal pacienților cu forma gravă, iar $Y$ în principal pacienților cu forma ușoară.

In [6]:
import pandas as pd

ex = pd.DataFrame({
    "Medicament": ["X", "X", "Y", "Y"],
    "Forma": ["usoara", "grava", "usoara", "grava"],
    "Succes": [19, 40, 180, 3],
    "Total": [20, 180, 200, 20],
})
ex["Rata"] = ex["Succes"] / ex["Total"]

pe_forma = ex.pivot(index="Forma", columns="Medicament", values="Rata")
print("Rata pe forma bolii")
print(pe_forma.round(4))

global_ex = ex.groupby("Medicament")[["Succes", "Total"]].sum()
global_ex["Rata globala"] = global_ex["Succes"] / global_ex["Total"]
print("\nRata globala")
print(global_ex.round(4))

pondere_ex = ex.pivot(index="Forma", columns="Medicament", values="Total")
print("\nPonderea formelor in fiecare grup de tratament")
print((pondere_ex / pondere_ex.sum()).round(4))

print("\nX mai bun in fiecare forma:", bool((pe_forma["X"] > pe_forma["Y"]).all()))
print("Y mai bun global:", bool(global_ex.loc["Y", "Rata globala"] > global_ex.loc["X", "Rata globala"]))

Rata pe forma bolii
Medicament       X     Y
Forma                   
grava       0.2222  0.15
usoara      0.9500  0.90

Rata globala
            Succes  Total  Rata globala
Medicament                             
X               59    200        0.2950
Y              183    220        0.8318

Ponderea formelor in fiecare grup de tratament
Medicament    X       Y
Forma                  
grava       0.9  0.0909
usoara      0.1  0.9091

X mai bun in fiecare forma: True
Y mai bun global: True
